# Data ingestion from a Unity Catalog volume and transformations with DataFrames

This notebook shows you how to load data from a Unity Catalog volume and how to transform it using the Apache Spark Python (PySpark) DataFrame API, including the following steps:

- Define variables and copy public baby names data into a Unity Catalog volume
- Load data into a DataFrame from the CSV file in the Unity Catalog volume
- View and interact with a DataFrame, explore the data
- Save the DataFrame

## Step 1. Define variables and load CSV file into your volume

Define variables:

In [0]:
download_url = "https://health.data.ny.gov/api/views/jxy9-yhdk/rows.csv"
file_name = "rows.csv"

catalog = "databricks_training"
schema = "demo"
volume = "baby_names"
path_volume = f"/Volumes/{catalog}/{schema}/{volume}"

table_name = "baby_names_table"
path_table = f"{catalog}.{schema}"

print(path_table)
print(path_volume)

Copy the `rows.csv` file from [health.data.ny.gov](https://health.data.ny.gov/) into your Unity Catalog volume using the Databricks `dbutuils` command

In [0]:
dbutils.fs.cp(f"{download_url}", f"{path_volume}/{file_name}")

## Step 2: Load data into a DataFrame from the CSV file

In [0]:
df_csv = spark.read.csv(f"{path_volume}/{file_name}",
    header=True,
    inferSchema=True,
    sep=",")
display(df_csv)

In [0]:
df_csv.count()

Print the schema of the DataFrame:

In [0]:
df_csv.printSchema()

## Step 3: Data exploration

Rename the `First Name` column to `First_Name` and the `Count` column to `Frequency`:

In [0]:
df_csv = df_csv.withColumnRenamed("First Name", "First_Name")
df_csv = df_csv.withColumnRenamed("Count", "Frequency")
df_csv.printSchema()

Only display the baby names that are chosen more than 50 times, using either Spark's `filter() ` or `.where()` method:

In [0]:
display(df_csv.filter(df_csv["Frequency"] > 50))
# display(df_csv.where(df_csv["Frequency"] > 50))

Display the most common names along with their frequencies (aggregated across all counties), sorted in descending order and create a word cloud visualization displaying the most frequent names.

In [0]:
from pyspark.sql.functions import desc, sum

display(
    df_csv.groupBy("First_Name")
    .agg(sum("Frequency").alias("Total_Frequency"))
    .orderBy(desc("Total_Frequency"))
)

Databricks visualization. Run in Databricks to view.

Create a new DataFrame with a subset of the data, including only girl names from 2009 that were chosen more than 100 times across different counties.

In [0]:
subset_df = (
    df_csv.filter(
        (df_csv["Year"] == 2009) &
        (df_csv["Frequency"] > 100) &
        (df_csv["Sex"] == "F")
    )
    .select("First_Name", "County", "Frequency")
    .orderBy(desc("Frequency"))
)
display(subset_df)

## Step 4: Save the DataFrame

#### Save to a table
Save the contents of the DataFrame to a table using the variables you defined at the start and check that the table appeared in the specified catalog and schema.

In [0]:
df_csv.write.mode("overwrite").saveAsTable(f"{path_table}.{table_name}")

#### Save to JSON files

Save the DataFrame to a directory of JSON files (`json_data`) in your Unity Catalog volume created previously and check that the `json_data` folder appeared in your Unity Catalog volume.

In [0]:
df_csv.write.format("json").mode("overwrite").save(f"{path_volume}/json_data")

Read and display the JSON files you saved in the previous example:

In [0]:
display(spark.read.format("json").json(f"{path_volume}/json_data"))